# Análise exploratória — Problemas no domicílio nas 12 maiores capitais (POF 2002-2003)
**Autor:** Alexandre Souza de Abreu Junior · **Instituição:** Centro Universitário La Salle do Rio de Janeiro  
**Curso:** Bacharelado em Sistemas de Informação · **Disciplina:** Linguagens de Programação (6º período, 2026.2) · **Professor:** Alexandre Neves Louzada

Notebook de exploração e validação. A lógica reutilizável está em `src/etl.py` e `src/analise.py`.
Execute a partir da **raiz do repositório** (a célula abaixo ajusta isso se você abrir dentro de `notebooks/`).

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
import pandas as pd
from src import etl, analise
pd.set_option("display.width", 160)

## 1. Dado bruto (como vem da planilha)

In [2]:
raw = etl.read_raw(etl.RAW_DEFAULT)
raw.iloc[:14, :6]

,0,1,2,3,4,5
0,Armazém de Dados,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,"Tabela 1461 - Famílias, por existência de prob...",NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,12 maiores capitais,NaN,NaN,NaN,NaN,NaN
5,NaN,Pouco espaço,NaN,Rua ou vizinhos barulhentos,NaN,Casa escura
6,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,Sim,Não,Sim,Não,Sim


## 2. ETL: formato tidy + validação

In [3]:
df = etl.run()
df.head(10)

ETL concluído: {'linhas': 96, 'capitais': 12, 'problemas': 8, 'nulos': 0}


,capital,regiao,problema,problema_original,pct_sim,pct_nao,rank_capital_no_problema,ano_referencia
0,Belém,Norte,barulho,Rua ou vizinhos barulhentos,44.20,55.80,1,2002-2003
1,Belo Horizonte,Sudeste,barulho,Rua ou vizinhos barulhentos,43.99,56.01,2,2002-2003
2,Porto Alegre,Sul,barulho,Rua ou vizinhos barulhentos,39.02,60.98,3,2002-2003
3,Salvador,Nordeste,barulho,Rua ou vizinhos barulhentos,38.80,61.20,4,2002-2003
4,São Paulo,Sudeste,barulho,Rua ou vizinhos barulhentos,37.44,62.56,5,2002-2003
5,Recife,Nordeste,barulho,Rua ou vizinhos barulhentos,32.26,67.74,6,2002-2003
6,Manaus,Norte,barulho,Rua ou vizinhos barulhentos,29.54,70.46,7,2002-2003
7,Fortaleza,Nordeste,barulho,Rua ou vizinhos barulhentos,28.84,71.16,8,2002-2003
8,Curitiba,Sul,barulho,Rua ou vizinhos barulhentos,27.19,72.81,9,2002-2003
9,Rio de Janeiro,Sudeste,barulho,Rua ou vizinhos barulhentos,26.57,73.43,10,2002-2003


In [4]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   capital                   96 non-null     str    
 1   regiao                    96 non-null     str    
 2   problema                  96 non-null     str    
 3   problema_original         96 non-null     str    
 4   pct_sim                   96 non-null     float64
 5   pct_nao                   96 non-null     float64
 6   rank_capital_no_problema  96 non-null     int64  
 7   ano_referencia            96 non-null     str    
dtypes: float64(2), int64(1), str(5)
memory usage: 6.1 KB


capital                     0
regiao                      0
problema                    0
problema_original           0
pct_sim                     0
pct_nao                     0
rank_capital_no_problema    0
ano_referencia              0
dtype: int64

## 3. RQ1 — Prevalência e dispersão por problema

In [5]:
w = analise.matriz(df)
desc = w.describe().T
desc["cv"] = desc["std"] / desc["mean"]
desc.sort_values("mean", ascending=False).round(2)

,count,mean,std,min,25%,50%,75%,max,cv
problema,,,,,,,,,
violencia,12.0,45.28,9.34,24.14,41.62,44.60,49.40,61.08,0.21
pouco_espaco,12.0,43.60,6.60,32.09,37.68,45.74,48.93,51.63,0.15
barulho,12.0,32.80,7.71,20.66,27.04,30.90,38.85,44.20,0.24
umidade,12.0,31.05,12.74,8.62,23.35,32.38,38.81,50.70,0.41
goteira,12.0,30.00,15.06,3.62,21.58,23.67,41.41,54.91,0.50
madeira_deteriorada,12.0,27.94,13.79,9.30,18.47,26.18,35.41,56.81,0.49
poluicao,12.0,24.94,8.14,12.76,19.52,23.87,31.88,38.11,0.33
casa_escura,12.0,19.58,6.68,5.29,17.71,20.56,23.09,29.31,0.34


## 4. RQ2 — Índice médio por capital e por região

In [6]:
indice = w.mean(axis=1).sort_values(ascending=False)
indice.round(2)

capital
Belém             46.60
Fortaleza         39.06
Recife            37.86
Manaus            35.87
Salvador          34.17
Porto Alegre      32.97
Belo Horizonte    32.50
São Paulo         32.18
Goiânia           27.26
Rio de Janeiro    25.43
Curitiba          23.25
Brasília          15.65
dtype: float64

In [7]:
indice.groupby(indice.index.map(etl.REGIAO)).mean().sort_values(ascending=False).round(2)

capital
Norte           41.24
Nordeste        37.03
Sudeste         30.04
Sul             28.11
Centro-Oeste    21.45
dtype: float64

In [8]:
z = (w - w.mean()) / w.std()
z[z.abs() >= 1.8].stack().dropna().round(2)   # valores extremos

capital                      
Belém     madeira_deteriorada    2.09
Brasília  casa_escura           -2.14
          violencia             -2.26
dtype: float64

## 5. RQ3 — Correlação entre problemas

In [9]:
w.corr(method="spearman").round(2)

problema,barulho,casa_escura,goteira,madeira_deteriorada,poluicao,pouco_espaco,umidade,violencia
problema,,,,,,,,
barulho,1.00,0.14,0.50,0.55,0.58,0.59,0.45,0.69
casa_escura,0.14,1.00,0.74,0.78,0.36,0.45,0.77,0.35
goteira,0.50,0.74,1.00,0.83,0.45,0.77,0.82,0.44
madeira_deteriorada,0.55,0.78,0.83,1.00,0.39,0.70,0.85,0.47
poluicao,0.58,0.36,0.45,0.39,1.00,0.47,0.32,0.83
pouco_espaco,0.59,0.45,0.77,0.70,0.47,1.00,0.71,0.28
umidade,0.45,0.77,0.82,0.85,0.32,0.71,1.00,0.45
violencia,0.69,0.35,0.44,0.47,0.83,0.28,0.45,1.00


In [10]:
from scipy import stats
stats.spearmanr(w["goteira"], w["madeira_deteriorada"])

SignificanceResult(statistic=np.float64(0.8251748251748254), pvalue=np.float64(0.0009513628555555001))

## 6. Figuras do artigo

In [11]:
analise.main()   # grava PNGs e estatisticas.json em docs/assets/

Figuras e estatísticas geradas em docs/assets


## 7. Observações
- Dados agregados (n = 12 capitais): correlações são **ecológicas** e instáveis.
- Informante único por família; respostas binárias e autodeclaradas.